# Notebook 09: Embedding Evaluation

## Why This Matters
Picking the wrong evaluation metric is how you ship a model that looks good on paper and fails in production. Embedding models are especially prone to this because the dominant benchmark (STS-B Spearman correlation) measures sentence-pair similarity on a general English dataset — which may have nothing to do with your retrieval task on medical documents.

This notebook builds a rigorous evaluation toolkit: STS for text similarity, recall@k for retrieval, and the diagnostic checks that reveal when a model is overfit to benchmarks rather than genuinely useful.

## Structure
1. STS benchmarks — measuring text similarity quality
2. Retrieval evaluation — precision@k, recall@k, NDCG, MRR
3. Intrinsic vs extrinsic evaluation pitfalls
4. MTEB — the modern multi-task embedding benchmark
5. Building a domain-specific evaluation set
6. Bridge to dimensionality reduction

## What You'll Learn
- How Spearman correlation, precision@k, recall@k, NDCG, and MRR are computed
- Why STS-B scores don't predict retrieval quality (and when they do)
- How to build a held-out evaluation set that reflects your actual task
- What MTEB measures and how to interpret its leaderboard


## Background

### The benchmark trap

A model that scores 0.92 Spearman on STS-B may retrieve completely wrong documents for your legal search system. Why? STS-B pairs are general English sentences scored for semantic similarity by crowdworkers. Legal retrieval needs a model that distinguishes "breach of contract" from "tortious interference" — a distinction that may not appear in STS-B at all.

The right evaluation pipeline:
1. **Intrinsic:** measure on standard benchmarks (STS-B, MTEB) for comparability
2. **Extrinsic:** measure on your actual task (retrieval recall@k, classification accuracy)
3. **Domain:** measure on a held-out set drawn from your actual data distribution

If intrinsic and extrinsic scores disagree, trust extrinsic.

### Retrieval metrics

**Precision@k:** Of the top-k retrieved results, what fraction is relevant?
```
P@k = |{relevant in top-k}| / k
```

**Recall@k:** Of all relevant results, what fraction appears in the top-k?
```
R@k = |{relevant in top-k}| / |{all relevant}|
```

**NDCG@k:** Normalized Discounted Cumulative Gain — rewards relevant results appearing earlier.
```
DCG@k = Σ_{i=1}^{k} rel_i / log₂(i+1)
NDCG@k = DCG@k / IDCG@k   (IDCG = ideal DCG if perfectly ranked)
```

**MRR:** Mean Reciprocal Rank — average of 1/rank of first relevant result.
```
MRR = (1/|Q|) Σ_q 1/rank_q
```

For retrieval systems: **recall@k is usually the most important** metric — it measures how often the correct answer appears in what you show the user.


In [ ]:
# %pip install -q sentence-transformers torch numpy scikit-learn matplotlib datasets

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer, util
from scipy.stats import spearmanr, pearsonr

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer("all-MiniLM-L6-v2", device=device)
print(f"Model loaded on {device}")

## 1. STS Evaluation — Spearman Correlation

In [ ]:
# STS-B style evaluation set
sts_pairs = [
    ("A man is playing guitar.", "A person is playing a musical instrument.", 4.5),
    ("Two dogs play in the grass.", "Two cats rest on the couch.", 0.5),
    ("She is cooking dinner.", "The woman is preparing a meal.", 4.2),
    ("The economy grew by 3%.", "GDP increased significantly this quarter.", 3.8),
    ("He jumped over the fence.", "The fence was painted red.", 0.2),
    ("A child is reading a book.", "A kid is studying.", 3.5),
    ("The plane landed safely.", "The aircraft arrived at the airport.", 4.0),
    ("It is raining outside.", "The weather is cold and windy.", 1.5),
    ("The scientist discovered a new species.", "Researchers identified a previously unknown animal.", 4.3),
    ("She smiled at him.", "He frowned and looked away.", 0.8),
]

sents_a = [p[0] for p in sts_pairs]
sents_b = [p[1] for p in sts_pairs]
human   = np.array([p[2] for p in sts_pairs])

emb_a = model.encode(sents_a, convert_to_tensor=True)
emb_b = model.encode(sents_b, convert_to_tensor=True)
cosines = util.cos_sim(emb_a, emb_b).diagonal().cpu().numpy()

spearman, _ = spearmanr(human, cosines)
pearson, _  = pearsonr(human, cosines)

print(f"Spearman correlation: {spearman:.3f}")
print(f"Pearson correlation:  {pearson:.3f}")
print()
print(f"{'Human':>7}  {'Model':>7}  Pair")
print("-" * 70)
for (a, b, h), m in zip(sts_pairs, cosines):
    print(f"{h:>7.1f}  {m:>7.3f}  {a[:25]!r} ↔ {b[:25]!r}")

## 2. Retrieval Metrics — Precision@k, Recall@k, NDCG, MRR

In [ ]:
def precision_at_k(retrieved_labels: list, relevant_label: int, k: int) -> float:
    top_k = retrieved_labels[:k]
    return sum(1 for l in top_k if l == relevant_label) / k

def recall_at_k(retrieved_labels: list, all_relevant_labels: list, k: int) -> float:
    top_k = retrieved_labels[:k]
    n_relevant_total = sum(1 for l in all_relevant_labels if l == all_relevant_labels[0])
    n_relevant_in_k  = sum(1 for l in top_k if l == all_relevant_labels[0])
    return n_relevant_in_k / n_relevant_total if n_relevant_total > 0 else 0.0

def ndcg_at_k(retrieved_labels: list, relevant_label: int, k: int) -> float:
    dcg = sum((1.0 if retrieved_labels[i] == relevant_label else 0.0) / np.log2(i + 2)
              for i in range(min(k, len(retrieved_labels))))
    # Ideal: first result is always correct
    idcg = sum(1.0 / np.log2(i + 2) for i in range(min(k, 1)))
    return dcg / idcg if idcg > 0 else 0.0

def mean_reciprocal_rank(all_retrieved: list[list], all_relevant: list[int]) -> float:
    rrs = []
    for retrieved, relevant in zip(all_retrieved, all_relevant):
        rr = 0.0
        for rank, label in enumerate(retrieved, start=1):
            if label == relevant:
                rr = 1.0 / rank
                break
        rrs.append(rr)
    return float(np.mean(rrs))


# Simulate a retrieval task: encode a corpus, retrieve for each query
corpus_texts = [
    "Flash attention reduces memory from O(N²) to O(N) using tiling",
    "The KV cache stores key-value pairs to avoid recomputation",
    "Speculative decoding proposes tokens with a draft model",
    "Quantization reduces model precision to INT8 or INT4",
    "Continuous batching allows new requests to join in-flight batches",
    "PagedAttention uses virtual memory paging for KV cache",
    "RoPE encodes position by rotating queries and keys",
    "Prefix caching reuses KV cache across identical prefixes",
    "Temperature scales logits before sampling",
    "Top-p nucleus sampling uses the minimal token set above probability p",
]
corpus_labels = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

queries = [
    ("how does flash attention save memory", 0),
    ("what is the kv cache for", 1),
    ("how does speculative decoding work", 2),
    ("what is quantization", 3),
    ("continuous batching explained", 4),
]

corpus_embs = model.encode(corpus_texts, convert_to_tensor=True)

results = {'p@1':[], 'p@3':[], 'r@3':[], 'ndcg@3':[], 'rank':[]}
for query_text, true_label in queries:
    q_emb = model.encode([query_text], convert_to_tensor=True)
    sims = util.cos_sim(q_emb, corpus_embs)[0].cpu().numpy()
    ranked_labels = [corpus_labels[i] for i in np.argsort(sims)[::-1]]

    results['p@1'].append(precision_at_k(ranked_labels, true_label, 1))
    results['p@3'].append(precision_at_k(ranked_labels, true_label, 3))
    results['r@3'].append(recall_at_k(ranked_labels, [true_label]*len(corpus_texts), 3))
    results['ndcg@3'].append(ndcg_at_k(ranked_labels, true_label, 3))
    results['rank'].append(next((i+1 for i, l in enumerate(ranked_labels) if l == true_label), len(corpus_texts)))

print("Retrieval metrics across 5 queries:")
print(f"  Precision@1:  {np.mean(results['p@1']):.3f}")
print(f"  Precision@3:  {np.mean(results['p@3']):.3f}")
print(f"  Recall@3:     {np.mean(results['r@3']):.3f}")
print(f"  NDCG@3:       {np.mean(results['ndcg@3']):.3f}")
print(f"  MRR:          {1/np.mean(results['rank']):.3f}  (approx)")
print()
print("Per-query rank of correct result:")
for (q, _), rank in zip(queries, results['rank']):
    print(f"  rank={rank}  {q}")

## 3. The STS–Retrieval Disconnect

In [ ]:
# Demonstrate that STS score doesn't predict retrieval quality
# A model can score well on STS while failing at retrieval if it conflates
# semantically related but informationally distinct documents

# Simulate two models: one with good STS, one with good retrieval
print("Why STS-B score doesn't predict retrieval quality:")
print()
print("Scenario: legal document retrieval")
print("  Query: 'breach of contract remedies'")
print()
print("  Model A (high STS-B): returns documents about 'contract law' broadly")
print("  → Semantically similar, but not the specific information needed")
print()
print("  Model B (lower STS-B): returns 'expectation damages', 'specific performance'")
print("  → Lower surface-form similarity, but exactly what the user needs")
print()
print("STS-B measures: 'are these two sentences about the same thing?'")
print("Retrieval measures: 'does this document answer this specific question?'")
print()
print("These are different tasks. Use retrieval metrics for retrieval tasks.")
print()
print("Correlation between STS-B and retrieval recall@5 on BEIR benchmark:")
print("  Spearman ≈ 0.55  (moderate but far from 1.0)")
print("  → STS-B explains ~30% of variance in retrieval quality")
print("  → Use domain-specific retrieval eval whenever possible")

## 4. MTEB — Multi-Task Embedding Benchmark

In [ ]:
# MTEB overview — the current standard for comparing embedding models
# https://huggingface.co/spaces/mteb/leaderboard

mteb_tasks = {
    "Classification":   "Encode text, train linear classifier — measures semantic content",
    "Clustering":       "Cluster embeddings, measure cluster purity — measures grouping",
    "Pair Classification": "Binary: are these two texts equivalent?",
    "Reranking":        "Reorder retrieved docs by relevance — closer to real retrieval",
    "Retrieval":        "Retrieve from corpus via query — closest to production",
    "STS":              "Sentence pair similarity — Spearman with human judgments",
    "Summarization":    "Rank summaries by quality vs reference",
}

print("MTEB task categories:")
for task, desc in mteb_tasks.items():
    print(f"  {task:<22}: {desc}")

print()
print("Top models on MTEB (as of 2024):")
leaderboard = [
    ("text-embedding-3-large", "OpenAI API",   64.59, "N/A",   "API only"),
    ("e5-mistral-7b-instruct", "HuggingFace",  66.63, "7.1B",  "Strong overall"),
    ("bge-large-en-v1.5",      "HuggingFace",  64.23, "335M",  "Best non-LLM open model"),
    ("all-mpnet-base-v2",      "HuggingFace",  57.78, "109M",  "Reliable, fast"),
    ("all-MiniLM-L6-v2",       "HuggingFace",  56.26, "22M",   "Best speed/quality"),
]

print(f"{'Model':<30} {'Source':<14} {'MTEB Avg':>9} {'Params':>8}  Notes")
print("-" * 80)
for name, source, score, params, notes in leaderboard:
    print(f"{name:<30} {source:<14} {score:>9.2f} {params:>8}  {notes}")

print()
print("Running MTEB:")
print("  from mteb import MTEB")
print("  eval = MTEB(tasks=['STSBenchmark', 'MSMARCO'])")
print("  results = eval.run(model)")

## 5. Building a Domain Evaluation Set

In [ ]:
# The right approach: collect evaluation examples from YOUR domain
# This is more valuable than any general benchmark for production decisions

def build_retrieval_eval_set(queries_with_answers: list, corpus: list):
    """
    queries_with_answers: [(query_text, [relevant_doc_idx, ...]), ...]
    corpus: [doc_text, ...]
    Returns metrics for a given model.
    """
    corpus_embs = model.encode(corpus, convert_to_tensor=True)
    metrics = {'recall@1': [], 'recall@5': [], 'mrr': []}

    for query, relevant_ids in queries_with_answers:
        q_emb = model.encode([query], convert_to_tensor=True)
        sims  = util.cos_sim(q_emb, corpus_embs)[0].cpu().numpy()
        ranked = list(np.argsort(sims)[::-1])

        # Recall@k: did any relevant doc appear in top-k?
        metrics['recall@1'].append(any(r in ranked[:1] for r in relevant_ids))
        metrics['recall@5'].append(any(r in ranked[:5] for r in relevant_ids))

        # MRR: reciprocal rank of first relevant result
        for rank, idx in enumerate(ranked, 1):
            if idx in relevant_ids:
                metrics['mrr'].append(1.0 / rank)
                break
        else:
            metrics['mrr'].append(0.0)

    return {k: np.mean(v) for k, v in metrics.items()}


# Domain eval: LLM inference concepts
inference_corpus = [
    "Flash attention tiles Q, K, V to avoid materializing the full attention matrix",
    "KV cache stores key and value projections from past tokens to skip recomputation",
    "Speculative decoding: draft model proposes tokens, main model accepts or rejects",
    "INT8 quantization reduces weight storage from 4 bytes to 1 byte per parameter",
    "Continuous batching: insert new sequences into the batch as old ones finish",
    "PagedAttention stores KV cache in non-contiguous physical memory blocks",
    "Tensor parallelism splits weight matrices across multiple GPUs column-wise",
    "Beam search maintains B candidate sequences and picks the highest log-probability",
]

eval_queries = [
    ("memory efficient attention computation", [0]),
    ("avoid recomputing attention in generation", [1]),
    ("faster inference with smaller model", [2]),
    ("reduce model memory footprint", [3]),
    ("maximize GPU utilization serving multiple users", [4]),
]

metrics = build_retrieval_eval_set(eval_queries, inference_corpus)
print("Domain-specific retrieval evaluation (LLM inference):")
for metric, value in metrics.items():
    print(f"  {metric:<12}: {value:.3f}")

print()
print("This is the evaluation that matters for production — not STS-B.")
print("Collect 50-200 (query, relevant_doc) pairs from your domain and run this.")

## 6. Bridge to Dimensionality Reduction

Evaluation tells you how good your embeddings are. But high-dimensional embeddings (384, 768, 1536 dimensions) are hard to inspect, expensive to index, and slow to search.

Dimensionality reduction serves two purposes:
1. **Visualization:** project to 2D/3D to understand the geometric structure of your embedding space
2. **Compression:** reduce dimensions to speed up search and reduce storage without large quality loss

Notebook 10 covers PCA, UMAP, and t-SNE — when to use each, how they distort the original geometry, and how much dimensionality you can remove before retrieval quality degrades.


## Summary

| Metric | Measures | When to use |
|--------|----------|-------------|
| Spearman (STS) | Sentence similarity correlation | General model comparison |
| Precision@k | Fraction of top-k that is relevant | When false positives are costly |
| Recall@k | Fraction of relevant that appear in top-k | When missing results is costly |
| NDCG@k | Rank-weighted relevance | When position in results matters |
| MRR | Rank of first correct result | Q&A, single-answer retrieval |
| MTEB | Aggregate across 56 tasks | Cross-model comparison, leaderboard |

**Practical rule:** build a domain-specific recall@k evaluation set. It takes a day and saves weeks of debugging the wrong model.

**Next:** Notebook 10 — Dimensionality Reduction. PCA vs UMAP vs t-SNE, how much dimensionality you can remove before retrieval degrades, and interactive embedding visualization.
